# Unidad 2: Aprendizaje Automático 
## 🔧 Tuning de Hiperparámetros con GridSearchCV
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Bosque](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-abdurrahim-israfilov-537700999-18415049.jpg)

[Foto de Abdurrahim Israfilov](https://www.pexels.com/es-es/foto/blanco-y-negro-musica-piano-interior-18415049/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/13_RandomForest_Tuning.ipynb)


## 🎯 ¿Qué vamos a aprender?

En el notebook anterior entrenamos un Random Forest con sus **parámetros por defecto**. Ahora aprenderemos a **optimizar** esos parámetros de forma sistemática.

Al finalizar, vas a poder:
- ✅ Entender qué son los **hiperparámetros** y por qué es importante ajustarlos
- ✅ Usar **GridSearchCV** para explorar combinaciones de hiperparámetros automáticamente
- ✅ Aplicar **validación cruzada (k-Fold)** dentro del proceso de búsqueda
- ✅ Obtener el **mejor conjunto de hiperparámetros** para el modelo
- ✅ Elegir una **métrica de optimización** apropiada para el problema

---

## 🧠 Conceptos Clave

### 🔩 ¿Qué son los Hiperparámetros?

Los **hiperparámetros** son los parámetros de configuración del modelo que se definen **antes** del entrenamiento (a diferencia de los *parámetros*, que el modelo aprende durante el entrenamiento).

Para Random Forest, los principales hiperparámetros son:

| Hiperparámetro | Descripción | Valor por defecto |
|----------------|-------------|-------------------|
| `n_estimators` | Número de árboles | 100 |
| `max_features` | Variables a evaluar en cada split | `'sqrt'` |
| `max_depth` | Profundidad máxima de cada árbol | `None` (sin límite) |
| `min_samples_split` | Mínimo de muestras para dividir un nodo | 2 |
| `min_samples_leaf` | Mínimo de muestras en cada hoja | 1 |

### 🔍 ¿Qué es GridSearchCV?

**GridSearchCV** (Grid Search + Cross-Validation) realiza una **búsqueda exhaustiva** sobre una grilla de combinaciones de hiperparámetros, evaluando cada combinación con **validación cruzada**.

```
Grilla de hiperparámetros:
  n_estimators: [10, 20, 30, 50, 100]
                ↓
  Para cada valor → entrenar con k-Fold CV → calcular score
                ↓
  Devolver la combinación con mejor score
```

### 🔄 ¿Qué es la Validación Cruzada (k-Fold CV)?

En lugar de hacer una sola división train/test, **k-Fold Cross-Validation** divide el dataset en `k` partes iguales y repite el entrenamiento `k` veces, usando cada parte como conjunto de validación una vez.

```
k=5:  [Fold1] [Fold2] [Fold3] [Fold4] [Fold5]
  Iter 1: TRAIN  TRAIN  TRAIN  TRAIN  TEST   → score1
  Iter 2: TRAIN  TRAIN  TRAIN  TEST   TRAIN  → score2
  Iter 3: TRAIN  TRAIN  TEST   TRAIN  TRAIN  → score3
  Iter 4: TRAIN  TEST   TRAIN  TRAIN  TRAIN  → score4
  Iter 5: TEST   TRAIN  TRAIN  TRAIN  TRAIN  → score5
                         Score final = media(score1..5)
```

Esto da una estimación **más robusta** del rendimiento real del modelo.

> 📌 **Referencias:**
> - Scikit-learn: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
> - Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, Cap. 2. O'Reilly Media.
> - Scikit-learn: [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)

---

## 📦 Paso 1: Importar las Librerías

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

# 🔍 GridSearchCV: búsqueda en grilla de hiperparámetros con validación cruzada
from sklearn.model_selection import GridSearchCV

print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar el Dataset

Usamos el mismo dataset **Breast Cancer Wisconsin** que en el notebook anterior.

In [ ]:
# 📥 Cargar el dataset
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data['data'], columns=cancer_data['feature_names'])
df['target'] = cancer_data['target']

# 🎯 Separar features y target
X = df[cancer_data.feature_names].values
y = df['target'].values

print(f'📐 Dimensiones del dataset: {X.shape}')
print(f'🏷️  Distribución de clases: {dict(zip(*[cancer_data.target_names, pd.Series(y).value_counts().values]))}')

## 🗺️ Paso 3: Definir el Espacio de Hiperparámetros

Definimos la **grilla** de valores que GridSearchCV va a explorar para `n_estimators`:
- Range: 10, 20, 25, 30, 40, 50, 72, 75, 100 árboles

> 💡 **¿Por qué explorar `n_estimators`?** Es el hiperparámetro más importante de Random Forest: más árboles generalmente mejoran el rendimiento, pero aumentan el tiempo de entrenamiento. Existe un punto de **rendimiento decreciente** a partir del cual agregar más árboles no mejora significativamente el modelo.

> ⚠️ **Costo computacional**: GridSearchCV con `cv=5` y `n_estimators` = 9 valores → necesita entrenar **9 × 5 = 45 modelos**.

In [ ]:
# 🗺️ Definición del espacio de hiperparámetros a explorar
param_grid = {
    'n_estimators': [10, 20, 25, 30, 40, 50, 72, 75, 100],
}

total_combinaciones = len(param_grid['n_estimators'])
cv_folds = 5
print(f'🔢 Valores de n_estimators: {param_grid["n_estimators"]}')
print(f'📊 Combinaciones a evaluar: {total_combinaciones}')
print(f'🔄 Folds de CV: {cv_folds}')
print(f'🤖 Total de modelos a entrenar: {total_combinaciones * cv_folds}')

## 🤖 Paso 4: Configurar y Ejecutar GridSearchCV

### ¿Por qué `scoring='f1'`?

Usamos **F1** como métrica de optimización porque:
- El dataset está **ligeramente desbalanceado** (más benignos que malignos)
- F1 balancea Precision y Recall, siendo más informativo que Accuracy sola
- En problemas médicos, tanto los FP como los FN tienen consecuencias importantes

El parámetro `verbose=5` muestra el progreso de la búsqueda.

In [ ]:
# 🤖 Definición del modelo base
rf = RandomForestClassifier(random_state=150)

# 🔍 Configuración de GridSearchCV
# scoring='f1': métrica a optimizar
# cv=5: validación cruzada con 5 folds
# verbose=5: muestra el progreso
gs = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1',
    cv=10,
    verbose=2
)

print('🚀 Iniciando búsqueda de hiperparámetros...')
print('⏳ Esto puede tardar unos segundos...')
gs.fit(X, y)
print('\n✅ Búsqueda completada!')

## 🏆 Paso 5: Análisis de Resultados

In [ ]:
# 🏆 Mejores parámetros encontrados
print('=' * 50)
print('🏆 MEJORES HIPERPARÁMETROS')
print('=' * 50)
print(f'  n_estimators: {gs.best_params_["n_estimators"]}')
print(f'  Mejor F1 (CV): {gs.best_score_:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 📊 Visualización de resultados
results = gs.cv_results_
n_vals  = param_grid['n_estimators']
scores  = results['mean_test_score']
stds    = results['std_test_score']

best_idx = list(n_vals).index(gs.best_params_['n_estimators'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(n_vals)), scores, color='steelblue', edgecolor='black', alpha=0.8)
ax.bar(best_idx, scores[best_idx], color='forestgreen', edgecolor='black', label='Mejor valor')
ax.errorbar(range(len(n_vals)), scores, yerr=stds, fmt='none', color='black', capsize=4)

ax.set_xticks(range(len(n_vals)))
ax.set_xticklabels(n_vals)
ax.set_xlabel('n_estimators')
ax.set_ylabel('F1 Score (media k-Fold)')
ax.set_title('F1 Score según n_estimators — GridSearchCV (cv=5)', fontsize=13)
ax.set_ylim(min(scores) - 0.01, 1.01)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n💡 El número óptimo de árboles es {gs.best_params_["n_estimators"]} con F1={gs.best_score_:.4f}')

## 🔬 Paso 6: Tabla de Resultados Detallada

In [ ]:
# 📋 Tabla de resultados completa
resultados_df = pd.DataFrame({
    'n_estimators': results['param_n_estimators'],
    'F1_mean':      results['mean_test_score'].round(4),
    'F1_std':       results['std_test_score'].round(4),
    'rank':         results['rank_test_score']
}).sort_values('rank')

print('📋 Resultados de GridSearchCV (ordenados por rank):')
resultados_df

## 🏁 Conclusiones

En este notebook aprendimos:

1. 🔩 Los **hiperparámetros** se configuran antes del entrenamiento y afectan directamente el rendimiento del modelo.
2. 🔍 **GridSearchCV** automatiza la búsqueda exhaustiva sobre una grilla de combinaciones.
3. 🔄 Combinar GridSearch con **validación cruzada (k-Fold)** da estimaciones más robustas y evita overfitting al proceso de selección.
4. 📊 Usar la **métrica correcta** (F1, Precision, Recall) según el problema es fundamental.
5. ⚡ Existe un punto de rendimiento decreciente para `n_estimators`: más árboles no siempre significa mejor rendimiento.

### ➡️ Próximo notebook: Tuning con múltiples hiperparámetros simultáneos

---

## 📚 Referencias

- Scikit-learn: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- Scikit-learn: [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2nd ed.), Cap. 2. O'Reilly Media.
- Bergstra, J., & Bengio, Y. (2012). [Random Search for Hyper-Parameter Optimization](https://www.jmlr.org/papers/v13/bergstra12a.html). *JMLR*, 13, 281–305.